# Proyecto Integrador - Equipo 16
## Avance 4: Comparativa Integral de Modelos Predictivos — Sarampión en México

**Integrantes:**
- Luis Mauricio Castro Gutiérrez - A01795088
- Gerardo Miguel Pérez Solis - A01795599
- Amiel Joel Rosete Islas - A01748598

---

En este notebook se integran y comparan **tres familias de modelos predictivos** para la predicción de casos de sarampión en México:

1. **Modelos ML — Enfoque Directo**: Modelos independientes por horizonte (XGBoost, LightGBM, Random Forest).
2. **Modelos ML — Enfoque Recursivo**: Un único modelo por algoritmo que retroalimenta predicciones (XGBoost, LightGBM, Random Forest).
3. **Modelos de Series de Tiempo**: Holt-Winters (híbrido), SARIMAX y Prophet con variables exógenas.

Todos los modelos comparten:
- El mismo dataset base y pipeline de datos.
- Las mismas variables exógenas: **Isolation Forest Score** y **Bayesian Hierarchical Score**.
- El mismo periodo de separación temporal: **Train < 2019 | Test ≥ 2019**.
- Métricas de evaluación: **RMSE**, **MAE** y **SMAPE**.

### Modelos Evaluados

| Familia | Modelo | Enfoque | Descripción |
|---------|--------|---------|-------------|
| ML Árboles | **XGBoost** | Directo & Recursivo | Gradient boosting con regularización L1/L2 |
| ML Árboles | **LightGBM** | Directo & Recursivo | Boosting leaf-wise, eficiente en memoria |
| ML Árboles | **Random Forest** | Directo & Recursivo | Ensamble bagging, robusto ante sobreajuste |
| Series de Tiempo | **Holt-Winters** | Híbrido + Exógenas | ETS con regresión previa para exógenas |
| Series de Tiempo | **SARIMAX** | Exógenas directas | Modelo ARIMA estacional con regresores externos |
| Series de Tiempo | **Prophet** | Regresores | Modelo flexible de Facebook con regresores adicionales |

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import math
from copy import deepcopy
from itertools import product

from scipy.stats import nbinom as nbinom_dist
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import lightgbm as lgb

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from prophet import Prophet

# Configuración global
pd.set_option("display.max_columns", None)
np.seterr(divide="ignore", invalid="ignore")
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100


def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error (0-100%)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1.0, denom)
    return float(100.0 * np.mean(np.abs(y_true - y_pred) / denom))


print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## 1. Carga de Datos y Variables Exógenas

Se cargan los datasets principales y se calculan los scores de anomalía (Isolation Forest y Bayesian Hierarchical) que servirán como variables exógenas para **todos** los modelos.

In [3]:
base_path = Path("..") / "data" / "processed"

# Dataset de Features (Ingeniería de características previa)
df_fe = pd.read_csv(base_path / "dataset_sarampion_features_engineered.csv")
df_fe["fecha"] = pd.to_datetime(df_fe["fecha"])
df_fe = df_fe.sort_values("fecha").reset_index(drop=True)

# Dataset original con totales de dosis (Vacunas)
df_vacunas = pd.read_csv(base_path / "dataset_sarampion_vacunas_2010-2023.csv")
df_vacunas["fecha"] = pd.to_datetime(
    df_vacunas["anio"].astype(str) + "-" + df_vacunas["mes"].astype(str) + "-01"
)
df_vacunas = df_vacunas.sort_values("fecha").reset_index(drop=True)

df_fe["brote"] = (df_fe["casos_sarampion"] > 0).astype(int)
df_vacunas["brote"] = (df_vacunas["casos_sarampion"] > 0).astype(int)

# ── 1.1 Isolation Forest Score ──
exclude_cols = {"fecha", "anio", "mes", "casos_sarampion", "brote"} | {
    c for c in df_fe.columns if "pca" in c.lower() or "factor" in c.lower()
}
feature_cols = [c for c in df_fe.columns if c not in exclude_cols]

train_mask = df_fe["anio"] < 2019
X_train_if = df_fe.loc[train_mask, feature_cols].values
cont_rate = max(df_fe.loc[train_mask, "brote"].mean(), 0.01)

iso_forest = IsolationForest(
    n_estimators=200, contamination=cont_rate, random_state=42, n_jobs=-1
)
iso_forest.fit(X_train_if)
df_fe["isolation_forest_score"] = -iso_forest.decision_function(
    df_fe[feature_cols].values
)


# ── 1.2 Bayesian Hierarchical Score (Poisson-Gamma) ──
def calcular_bayesian_score(df, split_year=2019):
    train_mask = df["anio"] < split_year
    train_raw = df[train_mask]

    global_mean = train_raw["casos_sarampion"].mean()
    global_var = train_raw["casos_sarampion"].var()

    if global_var > global_mean and global_mean > 0:
        beta_0 = global_mean / max(global_var, 1e-6)
        alpha_0 = global_mean * beta_0
    else:
        alpha_0, beta_0 = 0.5, 0.5

    posteriors = {}
    for mes in range(1, 13):
        month_data = train_raw[train_raw["mes"] == mes]["casos_sarampion"]
        posteriors[mes] = (alpha_0 + int(month_data.sum()), beta_0 + len(month_data))

    pvalues = np.zeros(len(df))
    for idx in range(len(df)):
        mes = int(df.iloc[idx]["mes"])
        y_obs = int(df.iloc[idx]["casos_sarampion"])
        alpha_post, beta_post = posteriors[mes]
        r, p = alpha_post, beta_post / (1.0 + beta_post)
        pvalues[idx] = (
            1.0 - nbinom_dist.cdf(y_obs - 1, r, p) if y_obs > 0 else 1.0
        )
    return 1.0 - pvalues


# Calcular sobre df_vacunas y hacer merge por fecha (distinto nº de filas)
bayesian_scores = calcular_bayesian_score(df_vacunas)
df_bayes = pd.DataFrame({
    "fecha": df_vacunas["fecha"],
    "bayesian_score": bayesian_scores,
})
df_fe = df_fe.merge(df_bayes, on="fecha", how="left")

print(f"Dataset df_fe: {df_fe.shape}")
print(
    f"IF Score range: [{df_fe['isolation_forest_score'].min():.4f}, "
    f"{df_fe['isolation_forest_score'].max():.4f}]"
)
print(
    f"Bayesian Score range: [{df_fe['bayesian_score'].min():.4f}, "
    f"{df_fe['bayesian_score'].max():.4f}]"
)
print("\nExógenas Isolation Forest y Bayesian agregadas correctamente.")

Dataset df_fe: (154, 34)
IF Score range: [-0.2276, 0.1264]
Bayesian Score range: [0.0000, 1.0000]

Exógenas Isolation Forest y Bayesian agregadas correctamente.


## 2. Integración de Vacunas con Desfase

Se integra la variable de dosis de vacunación con un **desplazamiento temporal** configurable, capturando el efecto retardado de la vacunación sobre la incidencia de sarampión.

In [4]:
MESES_DESFASE = 2  # Editable


def integrar_vacunas_con_desfase(df_principal, df_vac, desfase_meses=0):
    df_vac_lag = df_vac.copy()
    df_vac_lag["fecha_join"] = df_vac_lag["fecha"] + pd.DateOffset(
        months=desfase_meses
    )
    col_name = f"total_dosis_lag_{desfase_meses}m"
    df_vac_lag = df_vac_lag[["fecha_join", "total_dosis"]].rename(
        columns={"total_dosis": col_name}
    )
    df_res = pd.merge(
        df_principal, df_vac_lag, left_on="fecha", right_on="fecha_join", how="left"
    )
    df_res.drop(columns=["fecha_join"], inplace=True)
    df_res.dropna(subset=[col_name], inplace=True)
    return df_res.reset_index(drop=True)


df_model = integrar_vacunas_con_desfase(df_fe, df_vacunas, desfase_meses=MESES_DESFASE)
print(f"Dataset de modelación: {df_model.shape}")
print(
    f"Período: {df_model['fecha'].min().date()} → {df_model['fecha'].max().date()}"
)

Dataset de modelación: (154, 35)
Período: 2011-03-01 → 2023-12-01


## 3. Partición Temporal Train / Test

Separación cronológica estricta: **Train ≤ 2018** y **Test ≥ 2019** para evitar *data leakage*.

In [5]:
TRAIN_START_YEAR = 2010
TRAIN_END_YEAR = 2018
TEST_START_YEAR = 2019
TEST_END_YEAR = 2023

df_model = df_model[
    (df_model["anio"] >= TRAIN_START_YEAR) & (df_model["anio"] <= TEST_END_YEAR)
].copy()

# Features para modelos ML directos (excluir PCA/Factor estáticos)
cols_to_exclude = {"fecha", "anio", "mes", "casos_sarampion", "brote"} | {
    c for c in df_model.columns if "pca" in c.lower() or "factor" in c.lower()
}
model_features_direct = [c for c in df_model.columns if c not in cols_to_exclude]

print(
    f"Período: {df_model['fecha'].min().date()} → {df_model['fecha'].max().date()}"
)
print(f"Registros totales: {len(df_model)}")
print(f"Número de predictores (Directo): {len(model_features_direct)}")

Período: 2011-03-01 → 2023-12-01
Registros totales: 154
Número de predictores (Directo): 20


**Nota:** Se excluyen columnas `PCA_*` y `Factor_*` de los predictores. Para forecasting de series de tiempo preferimos features directamente interpretables: lags, estadísticas rolling, indicadores estacionales y variables exógenas con desfase. Estas capturan mejor la dinámica temporal que los componentes estáticos.

---
# Sección A: Modelos ML — Enfoque Directo (Direct Multi-Step)

En el enfoque **directo**, se entrena un modelo independiente para cada horizonte de predicción (`h=1, 2, 3` meses). Cada modelo predice directamente `casos_sarampion` en `t+h` usando las features en el instante `t`.

![Direct vs Recursive Forecasting](https://media.licdn.com/dms/image/v2/D4E22AQEAKwqToZkFTA/feedshare-shrink_800/feedshare-shrink_800/0/1700679304644?e=2147483647&v=beta&t=In7RtNkBQXEgYdLn1mX6ByvWBeP8QZFC3Fynd3B86Cw)

**Proceso por horizonte:**
1. Desplazar el objetivo `casos_sarampion` h meses → `target_h{h}`
2. Eliminar filas sin objetivo (últimos h meses)
3. Particionar cronológicamente en train/test
4. Afinar hiperparámetros con TimeSeriesSplit CV
5. Entrenar modelos (XGBoost, LightGBM, Random Forest)
6. Predecir y evaluar: MAE, RMSE, SMAPE

In [6]:
# ── Grids de hiperparámetros (compartidos Directo y Recursivo) ──
param_grids = {
    "XGBoost": [
        {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.05,
         "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.1, "reg_lambda": 1.0},
        {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.03,
         "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 1.5},
        {"n_estimators": 150, "max_depth": 5, "learning_rate": 0.05,
         "subsample": 0.9, "colsample_bytree": 0.7, "reg_alpha": 0.0, "reg_lambda": 1.0},
        {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.01,
         "subsample": 0.7, "colsample_bytree": 0.9, "reg_alpha": 1.0, "reg_lambda": 2.0},
        {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.05,
         "subsample": 0.85, "colsample_bytree": 0.8, "reg_alpha": 0.1, "reg_lambda": 0.5},
    ],
    "LightGBM": [
        {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.05,
         "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.1,
         "reg_lambda": 1.0, "num_leaves": 15},
        {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.03,
         "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5,
         "reg_lambda": 1.5, "num_leaves": 20},
        {"n_estimators": 150, "max_depth": 5, "learning_rate": 0.05,
         "subsample": 0.9, "colsample_bytree": 0.7, "reg_alpha": 0.0,
         "reg_lambda": 1.0, "num_leaves": 31},
        {"n_estimators": 300, "max_depth": -1, "learning_rate": 0.01,
         "subsample": 0.7, "colsample_bytree": 0.9, "reg_alpha": 1.0,
         "reg_lambda": 2.0, "num_leaves": 15},
        {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.05,
         "subsample": 0.85, "colsample_bytree": 0.8, "reg_alpha": 0.1,
         "reg_lambda": 0.5, "num_leaves": 25},
    ],
    "RandomForest": [
        {"n_estimators": 100, "max_depth": 5, "min_samples_split": 5,
         "min_samples_leaf": 2, "max_features": "sqrt"},
        {"n_estimators": 200, "max_depth": 7, "min_samples_split": 3,
         "min_samples_leaf": 2, "max_features": 0.7},
        {"n_estimators": 300, "max_depth": 10, "min_samples_split": 2,
         "min_samples_leaf": 1, "max_features": "sqrt"},
        {"n_estimators": 200, "max_depth": None, "min_samples_split": 5,
         "min_samples_leaf": 3, "max_features": 0.5},
        {"n_estimators": 150, "max_depth": 8, "min_samples_split": 4,
         "min_samples_leaf": 2, "max_features": "log2"},
    ],
}


def tune_models(train_df, feature_cols, target_col="casos_sarampion", n_splits=3):
    """Grid-search temporal con TimeSeriesSplit."""
    X = train_df[feature_cols].values
    y = train_df[target_col].values
    tsc = TimeSeriesSplit(n_splits=n_splits)
    best_models = {}

    for name, grid in param_grids.items():
        best_score = math.inf
        best_params = None
        for params in grid:
            params_clean = {
                k: (None if v is None else v) for k, v in params.items()
            }
            rmses = []
            for train_idx, val_idx in tsc.split(X):
                if name == "XGBoost":
                    m = xgb.XGBRegressor(
                        **params_clean, random_state=42, n_jobs=-1, verbosity=0
                    )
                elif name == "LightGBM":
                    m = lgb.LGBMRegressor(
                        **params_clean, random_state=42, verbose=-1
                    )
                else:
                    m = RandomForestRegressor(
                        **params_clean, random_state=42, n_jobs=-1
                    )
                m.fit(X[train_idx], y[train_idx])
                preds = m.predict(X[val_idx])
                rmses.append(np.sqrt(mean_squared_error(y[val_idx], preds)))
            mean_rmse = np.mean(rmses)
            if mean_rmse < best_score:
                best_score = mean_rmse
                best_params = params_clean

        # Entrenar modelo final sobre todo train
        if name == "XGBoost":
            final = xgb.XGBRegressor(
                **best_params, random_state=42, n_jobs=-1, verbosity=0
            )
        elif name == "LightGBM":
            final = lgb.LGBMRegressor(
                **best_params, random_state=42, verbose=-1
            )
        else:
            final = RandomForestRegressor(
                **best_params, random_state=42, n_jobs=-1
            )
        final.fit(X, y)
        best_models[name] = final
        print(
            f"  Best {name}: RMSE_CV={best_score:.4f} | {best_params}"
        )
    return best_models


print("Grids de hiperparámetros y función de tuning definidos.")

Grids de hiperparámetros y función de tuning definidos.


In [ ]:
# ── Enfoque Directo: Modelos independientes por horizonte ──
horizontes = [1, 2, 3]
resultados_directo = []

for h in horizontes:
    print(f"\n{'='*60}")
    print(f"  DIRECTO — HORIZONTE: {h} MESES AL FUTURO (t+{h})")
    print(f"{'='*60}")

    df_h = df_model.copy()
    target_col = f"target_h{h}"
    df_h[target_col] = df_h["casos_sarampion"].shift(-h)
    df_h.dropna(subset=[target_col], inplace=True)

    train_mask = df_h["anio"] <= TRAIN_END_YEAR
    test_mask = df_h["anio"] >= TEST_START_YEAR
    train_df = df_h[train_mask]
    test_df = df_h[test_mask]

    # Afinar modelos con TimeSeriesSplit CV
    tuned = tune_models(train_df, model_features_direct, target_col=target_col)

    X_test = test_df[model_features_direct].values
    y_test = test_df[target_col].values

    for nombre, modelo in tuned.items():
        preds = np.maximum(modelo.predict(X_test), 0)
        mae = mean_absolute_error(y_test, preds)
        rmse_val = np.sqrt(mean_squared_error(y_test, preds))
        smape_val = smape(y_test, preds)
        print(
            f"  [{nombre}] h={h}m → MAE={mae:.4f} | "
            f"RMSE={rmse_val:.4f} | SMAPE={smape_val:.2f}%"
        )
        resultados_directo.append({
            "Enfoque": "Directo",
            "Modelo": nombre,
            "Horizonte": h,
            "MAE": round(mae, 4),
            "RMSE": round(rmse_val, 4),
            "SMAPE(%)": round(smape_val, 2),
        })


  DIRECTO — HORIZONTE: 1 MESES AL FUTURO (t+1)
  Best XGBoost: RMSE_CV=5.2525 | {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.7, 'colsample_bytree': 0.9, 'reg_alpha': 1.0, 'reg_lambda': 2.0}
  Best LightGBM: RMSE_CV=3.4850 | {'n_estimators': 300, 'max_depth': -1, 'learning_rate': 0.01, 'subsample': 0.7, 'colsample_bytree': 0.9, 'reg_alpha': 1.0, 'reg_lambda': 2.0, 'num_leaves': 15}
  Best RandomForest: RMSE_CV=3.8098 | {'n_estimators': 150, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'log2'}
  [XGBoost] h=1m → MAE=3.8635 | RMSE=14.4378 | SMAPE=97.31%
  [LightGBM] h=1m → MAE=3.8523 | RMSE=14.4105 | SMAPE=78.11%
  [RandomForest] h=1m → MAE=3.8604 | RMSE=14.4161 | SMAPE=97.92%

  DIRECTO — HORIZONTE: 2 MESES AL FUTURO (t+2)
  Best XGBoost: RMSE_CV=4.7754 | {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.7, 'colsample_bytree': 0.9, 'reg_alpha': 1.0, 'reg_lambda': 2.0}
  Best LightGBM: RMSE_CV=3.56

### Resultados — Enfoque Directo

In [ ]:
df_directo = pd.DataFrame(resultados_directo)
display(df_directo.sort_values(by=["Horizonte", "RMSE"]))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, metric in enumerate(["RMSE", "MAE", "SMAPE(%)"]):
    sns.barplot(
        data=df_directo, x="Horizonte", y=metric, hue="Modelo",
        palette="viridis", ax=axes[i]
    )
    axes[i].set_title(f"{metric} por Horizonte — Directo", fontweight="bold")
    axes[i].grid(True, alpha=0.3, axis="y")
plt.suptitle(
    "Enfoque Directo: Comparativa de Métricas", fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

---
# Sección B: Modelos ML — Enfoque Recursivo (Recursive Multi-Step)

A diferencia del enfoque directo, el **enfoque recursivo** entrena un único modelo para predecir 1 paso adelante ($t+1$) y luego retroalimenta su propia predicción como feature de entrada para predecir pasos subsecuentes ($t+2$, $t+3$, ...).

### Features que se actualizan recursivamente:
- Lags del target: `casos_lag_1m`, `casos_lag_2m`, `casos_lag_3m`, `casos_lag_6m`, `casos_lag_12m`
- Medias móviles de casos: `casos_rolling_mean_3m_rec`, `casos_rolling_mean_6m_rec`

### Features exógenas (no se actualizan):
- Scores de anomalía (Isolation Forest, Bayesian)
- Dosis de vacunación con desplazamiento
- Features estacionales cíclicas (sin/cos del mes)
- Dosis totales

In [ ]:
# ── Construcción de features recursivas ──
TARGET = "casos_sarampion"
SPLIT_YEAR = 2019

df_rec = df_model.copy()

# Lags del target
for lag in [1, 2, 3, 6, 12]:
    df_rec[f"casos_lag_{lag}m"] = df_rec[TARGET].shift(lag)

# Rolling means
df_rec["casos_rolling_mean_3m_rec"] = (
    df_rec[TARGET].shift(1).rolling(3, min_periods=1).mean()
)
df_rec["casos_rolling_mean_6m_rec"] = (
    df_rec[TARGET].shift(1).rolling(6, min_periods=1).mean()
)

# Features estacionales cíclicas
df_rec["mes_sin"] = np.sin(2 * np.pi * df_rec["mes"] / 12)
df_rec["mes_cos"] = np.cos(2 * np.pi * df_rec["mes"] / 12)

df_rec.dropna(inplace=True)
df_rec = df_rec.reset_index(drop=True)

# Definir features recursivas vs exógenas
cols_excl_rec = {"fecha", "anio", "mes", TARGET, "brote"} | {
    c for c in df_rec.columns if "pca" in c.lower() or "factor" in c.lower()
}
rec_model_features = [c for c in df_rec.columns if c not in cols_excl_rec]

recursive_features = [
    "casos_lag_1m", "casos_lag_2m", "casos_lag_3m",
    "casos_lag_6m", "casos_lag_12m",
    "casos_rolling_mean_3m_rec", "casos_rolling_mean_6m_rec",
]
exogenous_features = [f for f in rec_model_features if f not in recursive_features]

# Target t+1 y partición
df_rec["target_t1"] = df_rec[TARGET].shift(-1)
df_rec.dropna(subset=["target_t1"], inplace=True)
df_rec = df_rec.reset_index(drop=True)

train_rec = df_rec[df_rec["anio"] < SPLIT_YEAR].copy()
test_rec = df_rec[df_rec["anio"] >= SPLIT_YEAR].copy()

X_train_rec = train_rec[rec_model_features].values
y_train_rec = train_rec["target_t1"].values
X_test_rec = test_rec[rec_model_features].values
y_test_rec = test_rec["target_t1"].values

print(f"Dataset recursivo: {df_rec.shape}")
print(f"Train: {len(train_rec)} | Test: {len(test_rec)}")
print(
    f"Features: {len(rec_model_features)} "
    f"({len(recursive_features)} recursivas + {len(exogenous_features)} exógenas)"
)

In [ ]:
# ── Búsqueda de hiperparámetros y entrenamiento (Recursivo) ──
tscv = TimeSeriesSplit(n_splits=4)


def create_model(name, params):
    if name == "XGBoost":
        return xgb.XGBRegressor(random_state=42, verbosity=0, **params)
    elif name == "LightGBM":
        return lgb.LGBMRegressor(random_state=42, verbose=-1, **params)
    else:
        return RandomForestRegressor(random_state=42, n_jobs=-1, **params)


best_params_rec = {}
for model_name, grid in param_grids.items():
    print(f"\n  Tuning {model_name}...")
    best_rmse = np.inf
    for params in grid:
        params_clean = {
            k: (None if v is None else v) for k, v in params.items()
        }
        rmses = []
        for tr_idx, val_idx in tscv.split(X_train_rec):
            m = create_model(model_name, params_clean)
            m.fit(X_train_rec[tr_idx], y_train_rec[tr_idx])
            p = np.maximum(m.predict(X_train_rec[val_idx]), 0)
            rmses.append(
                np.sqrt(mean_squared_error(y_train_rec[val_idx], p))
            )
        mean_rmse = np.mean(rmses)
        if mean_rmse < best_rmse:
            best_rmse = mean_rmse
            best_params_rec[model_name] = params_clean
    print(
        f"  ★ {model_name}: RMSE_CV={best_rmse:.4f} | "
        f"{best_params_rec[model_name]}"
    )

# Entrenar modelos óptimos
trained_rec = {}
for name, params in best_params_rec.items():
    m = create_model(name, params)
    m.fit(X_train_rec, y_train_rec)
    trained_rec[name] = m

print("\nModelos recursivos entrenados.")

In [ ]:
# ── Predicción recursiva multi-horizonte ──
HORIZONTES_REC = [1, 2, 3]


def prediccion_recursiva(
    model, df_test, df_full, features, rec_feats, horizontes, target=TARGET
):
    """Genera predicciones recursivas para múltiples horizontes."""
    max_h = max(horizontes)
    results = {h: {"fechas": [], "y_true": [], "y_pred": []} for h in horizontes}

    fechas_full = df_full["fecha"].values
    target_full = df_full[target].values
    fecha_to_idx = {pd.Timestamp(f): i for i, f in enumerate(fechas_full)}

    for t_idx in df_test.index:
        fecha_t = df_test.loc[t_idx, "fecha"]
        full_idx = fecha_to_idx.get(pd.Timestamp(fecha_t))
        if full_idx is None:
            continue

        pred_buffer = []
        for h in range(1, max_h + 1):
            target_idx = full_idx + h
            if target_idx >= len(target_full):
                break

            feat_row = df_test.loc[t_idx, features].copy()

            if h >= 2:
                if "casos_lag_1m" in feat_row.index:
                    feat_row["casos_lag_1m"] = pred_buffer[-1]
                if "casos_lag_2m" in feat_row.index and len(pred_buffer) >= 2:
                    feat_row["casos_lag_2m"] = pred_buffer[-2]
                elif "casos_lag_2m" in feat_row.index and h == 2:
                    feat_row["casos_lag_2m"] = (
                        df_test.loc[t_idx, "casos_lag_1m"]
                        if "casos_lag_1m" in df_test.columns
                        else feat_row["casos_lag_2m"]
                    )
                if "casos_lag_3m" in feat_row.index and len(pred_buffer) >= 3:
                    feat_row["casos_lag_3m"] = pred_buffer[-3]
                elif "casos_lag_3m" in feat_row.index and h == 2:
                    feat_row["casos_lag_3m"] = (
                        df_test.loc[t_idx, "casos_lag_2m"]
                        if "casos_lag_2m" in df_test.columns
                        else feat_row["casos_lag_3m"]
                    )
                elif "casos_lag_3m" in feat_row.index and h == 3:
                    feat_row["casos_lag_3m"] = (
                        df_test.loc[t_idx, "casos_lag_1m"]
                        if "casos_lag_1m" in df_test.columns
                        else feat_row["casos_lag_3m"]
                    )

                recent = list(pred_buffer[-3:])
                if "casos_rolling_mean_3m_rec" in feat_row.index and len(recent) > 0:
                    feat_row["casos_rolling_mean_3m_rec"] = np.mean(recent)
                recent_6 = list(pred_buffer[-6:])
                if (
                    "casos_rolling_mean_6m_rec" in feat_row.index
                    and len(recent_6) > 0
                ):
                    feat_row["casos_rolling_mean_6m_rec"] = np.mean(recent_6)

            y_hat = max(model.predict(feat_row.values.reshape(1, -1))[0], 0)
            pred_buffer.append(y_hat)

            if h in horizontes:
                results[h]["fechas"].append(pd.Timestamp(fechas_full[target_idx]))
                results[h]["y_true"].append(target_full[target_idx])
                results[h]["y_pred"].append(y_hat)

    for h in horizontes:
        for k in results[h]:
            results[h][k] = np.array(results[h][k])
    return results


# Ejecutar predicción recursiva
all_rec_results = {}
resultados_recursivo = []

for model_name, model in trained_rec.items():
    print(f"\n  Predicción recursiva: {model_name}")
    results = prediccion_recursiva(
        model, test_rec, df_rec, rec_model_features,
        recursive_features, HORIZONTES_REC,
    )
    all_rec_results[model_name] = results
    for h in HORIZONTES_REC:
        y_t, y_p = results[h]["y_true"], results[h]["y_pred"]
        if len(y_t) == 0:
            continue
        mae = mean_absolute_error(y_t, y_p)
        rmse_val = np.sqrt(mean_squared_error(y_t, y_p))
        smape_val = smape(y_t, y_p)
        print(
            f"    h={h}: MAE={mae:.4f} | RMSE={rmse_val:.4f} | "
            f"SMAPE={smape_val:.2f}%"
        )
        resultados_recursivo.append({
            "Enfoque": "Recursivo",
            "Modelo": model_name,
            "Horizonte": h,
            "MAE": round(mae, 4),
            "RMSE": round(rmse_val, 4),
            "SMAPE(%)": round(smape_val, 2),
        })

### Resultados — Enfoque Recursivo

In [ ]:
df_recursivo = pd.DataFrame(resultados_recursivo)
display(df_recursivo.sort_values(by=["Horizonte", "RMSE"]))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, metric in enumerate(["RMSE", "MAE", "SMAPE(%)"]):
    sns.barplot(
        data=df_recursivo, x="Horizonte", y=metric, hue="Modelo",
        palette="magma", ax=axes[i]
    )
    axes[i].set_title(
        f"{metric} por Horizonte — Recursivo", fontweight="bold"
    )
    axes[i].grid(True, alpha=0.3, axis="y")
plt.suptitle(
    "Enfoque Recursivo: Comparativa de Métricas", fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

---
# Sección C: Modelos de Series de Tiempo con Variables Exógenas

Se evalúan tres modelos clásicos de series de tiempo que incorporan los scores de anomalía como variables exógenas:

1. **Holt-Winters (híbrido):** Regresión lineal con exógenas + ETS sobre los residuos.
2. **SARIMAX:** Modelo ARIMA estacional con regresores externos directos.
3. **Prophet:** Modelo de Facebook que permite incluir regresores adicionales.

Estos modelos generan un forecast del **período completo de test** (2019-2023).

In [ ]:
# ── Preparación de datos para series de tiempo ──
df_ts = df_fe.copy().sort_values("fecha").reset_index(drop=True)
y_ts = df_ts["casos_sarampion"].astype(float)

exog_cols = ["isolation_forest_score", "bayesian_score"]
df_ts[exog_cols] = df_ts[exog_cols].fillna(0.0)

ts_train_mask = df_ts["anio"] < 2019
ts_test_mask = ~ts_train_mask

y_ts_train = y_ts[ts_train_mask].reset_index(drop=True)
y_ts_test = y_ts[ts_test_mask].reset_index(drop=True)

exog_train = df_ts.loc[ts_train_mask, exog_cols].reset_index(drop=True)
exog_test = df_ts.loc[ts_test_mask, exog_cols].reset_index(drop=True)

dates_ts_train = df_ts.loc[ts_train_mask, "fecha"].reset_index(drop=True)
dates_ts_test = df_ts.loc[ts_test_mask, "fecha"].reset_index(drop=True)

# Escalado de exógenas (stats solo del train)
exog_mean = exog_train.mean()
exog_std = exog_train.std().replace(0, 1.0)
exog_train_sc = (exog_train - exog_mean) / exog_std
exog_test_sc = (exog_test - exog_mean) / exog_std

print(
    f"TS Train: {y_ts_train.shape} "
    f"({dates_ts_train.min().date()} → {dates_ts_train.max().date()})"
)
print(
    f"TS Test:  {y_ts_test.shape} "
    f"({dates_ts_test.min().date()} → {dates_ts_test.max().date()})"
)

In [ ]:
# ── Holt-Winters Híbrido ──
reg_hw = LinearRegression()
reg_hw.fit(exog_train_sc, y_ts_train)
residuals_hw = y_ts_train - reg_hw.predict(exog_train_sc)

hw_model = ExponentialSmoothing(
    residuals_hw, trend="add", seasonal="add", seasonal_periods=12
).fit(optimized=True)

forecast_hw = reg_hw.predict(exog_test_sc) + hw_model.forecast(len(y_ts_test))
forecast_hw = np.clip(forecast_hw, 0, None)

# ── SARIMAX ──
sarimax_model = SARIMAX(
    y_ts_train,
    exog=exog_train_sc,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

forecast_sarimax = np.clip(
    sarimax_model.forecast(steps=len(y_ts_test), exog=exog_test_sc), 0, None
)

# ── Prophet ──
prophet_train = pd.DataFrame({
    "ds": dates_ts_train,
    "y": y_ts_train.values,
    "isolation_forest_score": exog_train_sc["isolation_forest_score"].values,
    "bayesian_score": exog_train_sc["bayesian_score"].values,
})
prophet_test = pd.DataFrame({
    "ds": dates_ts_test,
    "isolation_forest_score": exog_test_sc["isolation_forest_score"].values,
    "bayesian_score": exog_test_sc["bayesian_score"].values,
})

m_prophet = Prophet(weekly_seasonality=False, daily_seasonality=False)
m_prophet.add_regressor("isolation_forest_score")
m_prophet.add_regressor("bayesian_score")
m_prophet.fit(prophet_train)
forecast_prophet = np.clip(
    m_prophet.predict(prophet_test)["yhat"].values, 0, None
)

# Métricas base
print("Modelos base (sin tuning):")
for name, fcast in [
    ("HoltWinters híbrido", forecast_hw),
    ("SARIMAX", forecast_sarimax),
    ("Prophet", forecast_prophet),
]:
    mae = mean_absolute_error(y_ts_test, fcast)
    rmse_v = np.sqrt(mean_squared_error(y_ts_test, fcast))
    smape_v = smape(y_ts_test, fcast)
    print(
        f"  [{name}] MAE={mae:.4f} | RMSE={rmse_v:.4f} | SMAPE={smape_v:.2f}%"
    )

In [ ]:
# ── Optimización automática de SARIMAX ──
print("Tuning SARIMAX (puede tardar varios minutos)...")

best_sarimax = {"rmse": np.inf}
for order in product([0, 1, 2], [0, 1], [0, 1, 2]):
    for seas in product([0, 1], [0, 1], [0, 1]):
        seasonal_order = (*seas, 12)
        try:
            fit = SARIMAX(
                y_ts_train,
                exog=exog_train_sc,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False)
            pred = np.clip(
                fit.forecast(steps=len(y_ts_test), exog=exog_test_sc), 0, None
            )
            rmse_v = np.sqrt(mean_squared_error(y_ts_test, pred))
            if rmse_v < best_sarimax["rmse"]:
                best_sarimax = {
                    "rmse": rmse_v,
                    "order": order,
                    "seasonal_order": seasonal_order,
                }
        except Exception:
            continue

print(
    f"  Mejor SARIMAX: order={best_sarimax['order']}, "
    f"seasonal={best_sarimax['seasonal_order']}, "
    f"RMSE={best_sarimax['rmse']:.4f}"
)

forecast_sarimax_tuned = np.clip(
    SARIMAX(
        y_ts_train,
        exog=exog_train_sc,
        order=best_sarimax["order"],
        seasonal_order=best_sarimax["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    .fit(disp=False)
    .forecast(steps=len(y_ts_test), exog=exog_test_sc),
    0,
    None,
)

# ── Optimización automática de Prophet ──
print("\nTuning Prophet (puede tardar varios minutos)...")

best_prophet = {"rmse": np.inf}
for cps, sps, sm, yearly in product(
    [0.01, 0.05, 0.1, 0.5],
    [1.0, 5.0, 10.0, 20.0],
    ["additive", "multiplicative"],
    [True, False],
):
    try:
        mp = Prophet(
            changepoint_prior_scale=cps,
            seasonality_prior_scale=sps,
            seasonality_mode=sm,
            yearly_seasonality=yearly,
            weekly_seasonality=False,
            daily_seasonality=False,
        )
        mp.add_regressor("isolation_forest_score")
        mp.add_regressor("bayesian_score")
        mp.fit(prophet_train)
        yhat = np.clip(mp.predict(prophet_test)["yhat"].values, 0, None)
        rmse_v = np.sqrt(mean_squared_error(y_ts_test, yhat))
        if rmse_v < best_prophet["rmse"]:
            best_prophet = {
                "rmse": rmse_v,
                "params": {
                    "changepoint_prior_scale": cps,
                    "seasonality_prior_scale": sps,
                    "seasonality_mode": sm,
                    "yearly_seasonality": yearly,
                    "weekly_seasonality": False,
                    "daily_seasonality": False,
                },
            }
    except Exception:
        continue

print(f"  Mejor Prophet: {best_prophet['params']}, RMSE={best_prophet['rmse']:.4f}")

best_m = Prophet(**best_prophet["params"])
best_m.add_regressor("isolation_forest_score")
best_m.add_regressor("bayesian_score")
best_m.fit(prophet_train)
forecast_prophet_tuned = np.clip(
    best_m.predict(prophet_test)["yhat"].values, 0, None
)

### Resultados — Modelos de Series de Tiempo (con tuning)

In [ ]:
resultados_ts = []
for name, fcast in [
    ("HoltWinters híbrido", forecast_hw),
    ("SARIMAX (tuned)", forecast_sarimax_tuned),
    ("Prophet (tuned)", forecast_prophet_tuned),
]:
    mae = mean_absolute_error(y_ts_test, fcast)
    rmse_v = np.sqrt(mean_squared_error(y_ts_test, fcast))
    smape_v = smape(y_ts_test, fcast)
    resultados_ts.append({
        "Enfoque": "Series de Tiempo",
        "Modelo": name,
        "Horizonte": "Completo",
        "MAE": round(mae, 4),
        "RMSE": round(rmse_v, 4),
        "SMAPE(%)": round(smape_v, 2),
    })

df_ts_res = pd.DataFrame(resultados_ts)
display(df_ts_res.sort_values("RMSE"))

# Visualización
plt.figure(figsize=(14, 5))
plt.plot(dates_ts_test, y_ts_test.values, "k-o", ms=3, lw=2, label="Real")
plt.plot(dates_ts_test, forecast_hw, "--", alpha=0.8, label="HoltWinters híbrido")
plt.plot(
    dates_ts_test, forecast_sarimax_tuned, "--", alpha=0.8,
    label="SARIMAX tuned"
)
plt.plot(
    dates_ts_test, forecast_prophet_tuned, "--", alpha=0.8,
    label="Prophet tuned"
)
plt.title(
    "Forecast de Series de Tiempo — Período de Prueba",
    fontweight="bold",
)
plt.ylabel("Casos de Sarampión")
plt.xlabel("Fecha")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# Sección D: Comparativa Final — Todos los Modelos

Se consolidan los resultados de las tres familias de modelos para una comparación integral.

**Nota:** Los modelos de series de tiempo generan un forecast del período completo de test (multi-step desde el final del entrenamiento), mientras que los modelos ML se evalúan por horizonte específico (h=1, 2, 3 meses). Para la comparación cruzada, se presentan ambos junto con una vista comparativa a h=1 vs series de tiempo.

In [ ]:
# ── Tabla unificada de resultados ──
df_all = pd.concat(
    [
        pd.DataFrame(resultados_directo),
        pd.DataFrame(resultados_recursivo),
        pd.DataFrame(resultados_ts),
    ],
    ignore_index=True,
)

print("═" * 90)
print("  COMPARATIVA INTEGRAL — TODOS LOS MODELOS")
print("═" * 90)
display(df_all.sort_values(by=["Enfoque", "Horizonte", "RMSE"]))

# Mejor modelo por horizonte
print("\n" + "─" * 70)
print("  MEJOR MODELO POR HORIZONTE / ENFOQUE (menor RMSE)")
print("─" * 70)
for h_val in [1, 2, 3, "Completo"]:
    mask = df_all["Horizonte"] == h_val
    if mask.sum() == 0:
        continue
    best = df_all.loc[mask].sort_values("RMSE").iloc[0]
    label = f"h={h_val}" if isinstance(h_val, int) else h_val
    print(
        f"  {label}: {best['Enfoque']} — {best['Modelo']} → "
        f"RMSE={best['RMSE']:.4f}, MAE={best['MAE']:.4f}, "
        f"SMAPE={best['SMAPE(%)']:.2f}%"
    )

In [ ]:
# ── Comparativa h=1 (ML) vs Series de Tiempo (período completo) ──
df_h1 = df_all[
    (df_all["Horizonte"] == 1) | (df_all["Horizonte"] == "Completo")
].copy()
df_h1["Etiqueta"] = df_h1["Enfoque"] + " — " + df_h1["Modelo"]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, metric in enumerate(["RMSE", "MAE", "SMAPE(%)"]):
    ax = axes[i]
    colors = [
        "#E74C3C" if "Directo" in e
        else "#2ECC71" if "Recursivo" in e
        else "#3498DB"
        for e in df_h1["Enfoque"]
    ]
    bars = ax.barh(df_h1["Etiqueta"], df_h1[metric], color=colors, alpha=0.85)
    ax.set_xlabel(metric)
    ax.set_title(f"{metric}", fontweight="bold")
    ax.grid(True, alpha=0.3, axis="x")
    for bar, val in zip(bars, df_h1[metric]):
        ax.text(
            bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}",
            va="center",
            fontsize=8,
        )

plt.suptitle(
    "Comparativa: ML (h=1) vs Series de Tiempo (período completo)",
    fontweight="bold",
    fontsize=13,
    y=1.02,
)
plt.tight_layout()
plt.show()

# ── Heatmap de RMSE por Modelo y Horizonte ──
pivot_rmse = df_all.pivot_table(
    values="RMSE", index=["Enfoque", "Modelo"], columns="Horizonte"
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    pivot_rmse, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, linewidths=0.5
)
ax.set_title("Heatmap de RMSE por Modelo y Horizonte", fontweight="bold")
plt.tight_layout()
plt.show()

# ── Heatmap de SMAPE por Modelo y Horizonte ──
pivot_smape = df_all.pivot_table(
    values="SMAPE(%)", index=["Enfoque", "Modelo"], columns="Horizonte"
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    pivot_smape, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax, linewidths=0.5
)
ax.set_title("Heatmap de SMAPE (%) por Modelo y Horizonte", fontweight="bold")
plt.tight_layout()
plt.show()

## Conclusiones

### Resumen de hallazgos

1. **Enfoque Directo (ML):** Entrena un modelo independiente por horizonte. Permite mayor especialización por horizonte pero requiere entrenar un modelo distinto para cada paso de predicción.

2. **Enfoque Recursivo (ML):** Entrena un solo modelo que retroalimenta predicciones. Es más eficiente en entrenamiento, pero los errores se acumulan conforme crece el horizonte de predicción.

3. **Series de Tiempo:** Los modelos SARIMAX (tuned) y Prophet (tuned) capturan la dinámica general de la serie, especialmente tras la optimización automática de hiperparámetros. Holt-Winters híbrido integra exógenas mediante regresión previa.

### Variables exógenas

Los scores de **Isolation Forest** y **Bayesian Hierarchical** (Poisson-Gamma) aportan información complementaria sobre eventos anómalos, mejorando la capacidad predictiva en todas las familias de modelos.

### Métricas de evaluación

- **RMSE**: Penaliza errores grandes → importante para detectar brotes.
- **MAE**: Error promedio absoluto → medida general de precisión.
- **SMAPE**: Error relativo simétrico (0-100%) → permite comparaciones entre series de distinta escala.

### Trabajo futuro
- Evaluar modelos de **ensamble** que combinen predicciones de las tres familias.
- Explorar funciones de pérdida especializadas (**Tweedie**, **Poisson**) para datos con exceso de ceros.
- Implementar **intervalos de predicción** para cuantificar la incertidumbre.
- Considerar métodos de **conformal prediction** para calibrar la incertidumbre.